In [11]:
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

True

In [12]:
from langchain.chat_models import init_chat_model

In [13]:
llm = init_chat_model(
    model="claude-haiku-4-5",
    temperature=0,
    max_tokens=400,  # 150 cortaba las respuestas a media frase
)

In [14]:
response = llm.invoke("Hola, ¿cómo estás?")
response.text

'¡Hola! Estoy bien, gracias por preguntar. 😊\n\n¿Y tú? ¿Cómo estás? ¿Hay algo en lo que pueda ayudarte?'

In [15]:
# sin tool
response = llm.invoke("¿qué clima hace en la ciudad de Bogotá")
response.pretty_print()

================================== Ai Message ==================================

# Clima en Bogotá

Bogotá tiene un clima **templado y frío** característico de las tierras altas andinas:

## Características principales:

- **Temperatura promedio**: 13-14°C (55-57°F)
- **Rango**: Mínimas de 7-8°C, máximas de 19-20°C
- **Altitud**: 2.640 metros sobre el nivel del mar (esto influye mucho en el clima)

## Estaciones:

- **Lluvias**: Abril-mayo y octubre-noviembre
- **Secas**: Diciembre-marzo y julio-agosto
- **Humedad**: Generalmente alta

## Recomendaciones:

- 🧥 Llevar abrigo o chaqueta (especialmente en las noches)
- ☔ En temporada de lluvias, paraguas
- 🕶️ Protector solar (la radiación UV es fuerte por la altitud)

¿Necesitas información más específica sobre el clima actual o para una fecha particular?


In [16]:
# sin tool
response = llm.invoke("Dime ¿qué productos orfreces en la tienda?")
response.pretty_print()

================================== Ai Message ==================================

# Productos en la tienda

No tengo acceso a información sobre una tienda específica. Para ayudarte mejor, necesito que me digas:

- **¿Cuál es el nombre de la tienda?**
- **¿Es una tienda física o en línea?**
- **¿En qué ciudad o país está ubicada?**

Con estos datos podré:
- Buscar información sobre sus productos
- Ayudarte a encontrar lo que necesitas
- Darte detalles sobre disponibilidad y precios

¿Puedes compartir más detalles? 😊


In [17]:
# sin tool con system propmt
system_prompt = """
Eres un asistente de ventas de una tienda de ropa. Tu tarea es responder a las preguntas de los clientes sobre los productos que ofrecemos.

Y tus productos son:
- Camisetas de algodón
- Pantalones vaqueros
- Chaquetas de cuero

"""

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "Dime ¿qué productos orfreces en la tienda?"},
]

response = llm.invoke(messages)
response.pretty_print()

================================== Ai Message ==================================

¡Hola! 👋 

En nuestra tienda ofrecemos los siguientes productos:

1. **Camisetas de algodón** - Cómodas y de alta calidad, perfectas para el día a día
2. **Pantalones vaqueros** - Clásicos y versátiles, ideales para cualquier ocasión
3. **Chaquetas de cuero** - Elegantes y duraderas, para un look sofisticado

¿Te interesa conocer más detalles sobre alguno de estos productos? Estoy aquí para ayudarte. 😊


In [18]:
# tool sin parametros
from langchain_core.tools import tool

@tool
def get_products():
    """
    Devuelve una lista de productos disponibles en la tienda.
    """
    return ["Camisetas de algodón", "Pantalones vaqueros", "Chaquetas de cuero"]

In [19]:
# tool con parametros

@tool("get_products", description="Devuelve una lista de productos disponibles en la tienda que sean menores o iguales al precio dado.")
def get_products(max_price: float):
    """
    Devuelve una lista de productos disponibles en la tienda que sean menores o iguales al precio dado.
    """
    products = {
        "Camisetas de algodón": 20.0,
        "Pantalones vaqueros": 50.0,
        "Chaquetas de cuero": 100.0
    }
    # devolver en texto
    return "\n".join([f"{product}: ${price}" for product, price in products.items()])

In [20]:
print(get_products.invoke({"max_price": 50.0}))

Camisetas de algodón: $20.0
Pantalones vaqueros: $50.0
Chaquetas de cuero: $100.0


In [21]:
# tool desde api rest
import requests
from langchain_core.tools import tool

# API de e-commerce con productos ficticios; no requiere token para consultarlos.
# Directorio: https://publicapi.dev/fake-store-api
# Documentación: https://github.com/keikaavousi/fake-store-api

@tool("get_products", description="Get the products that the store offers filtered by price")
def get_products():
    """
    Devuelve una lista de productos disponibles en la tienda que sean menores o iguales al precio dado.
    """
    response = requests.get("https://fakestoreapi.com/products", timeout=15)
    response.raise_for_status()
    products = response.json()
    return "\n".join([f"{product['title']}: ${product['price']}" for product in products])

In [22]:
print(get_products.invoke({}))

Fjallraven - Foldsack No. 1 Backpack, Fits 15 Laptops: $109.95
Mens Casual Premium Slim Fit T-Shirts : $22.3
Mens Cotton Jacket: $55.99
Mens Casual Slim Fit: $15.99
John Hardy Women's Legends Naga Gold & Silver Dragon Station Chain Bracelet: $695
Solid Gold Petite Micropave : $168
White Gold Plated Princess: $9.99
Pierced Owl Rose Gold Plated Stainless Steel Double: $10.99
WD 2TB Elements Portable External Hard Drive - USB 3.0 : $64
SanDisk SSD PLUS 1TB Internal SSD - SATA III 6 Gb/s: $109
Silicon Power 256GB SSD 3D NAND A55 SLC Cache Performance Boost SATA III 2.5: $109
WD 4TB Gaming Drive Works with Playstation 4 Portable External Hard Drive: $114
Acer SB220Q bi 21.5 inches Full HD (1920 x 1080) IPS Ultra-Thin: $599
Samsung 49-Inch CHG90 144Hz Curved Gaming Monitor (LC49HG90DMNXZA) – Super Ultrawide Screen QLED : $999.99
BIYLACLESEN Women's 3-in-1 Snowboard Jacket Winter Coats: $56.99
Lock and Love Women's Removable Hooded Faux Leather Moto Biker Jacket: $29.95
Rain Jacket Women Wind

In [23]:
# weather tool
import requests
from langchain_core.tools import tool

@tool("get_weather", description="Obtiene el clima actual de una ciudad.")
def get_weather(city: str) -> str:
    response = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": city, "count": 1},
        timeout=15,
    )
    response.raise_for_status()
    location = response.json()["results"][0]

    response = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": location["latitude"],
            "longitude": location["longitude"],
            "current_weather": True,
        },
        timeout=15,
    )
    response.raise_for_status()
    weather = response.json()["current_weather"]
    return f"El clima en {city} es de {weather['temperature']} °C con viento de {weather['windspeed']} km/h."



In [24]:
get_weather.invoke({"city": "Bogotá"})

'El clima en Bogotá es de 18.3 °C con viento de 13.2 km/h.'

In [27]:
# ahora llm + tool + system prompt

from langchain.messages import SystemMessage, HumanMessage

system_prompt = """
Eres un asistente de ventas de una tienda de ropa. Tu tarea es ayudar a los clientes a encontrar los productos que buscan y proporcionar información sobre el clima en diferentes ciudades.

Tus tools son:
- get_products: Devuelve una lista de productos disponibles en la tienda.
- get_weather: Obtiene el clima actual de una ciudad.
"""

messages = [
    SystemMessage(content=system_prompt),
    HumanMessage(content="Dime ¿qué productos ofreces en la tienda?"),
]

llm_tools = llm.bind_tools([get_products, get_weather])

In [30]:
response = llm_tools.invoke(messages)
response.pretty_print()

================================== Ai Message ==================================

[{'id': 'toolu_01CdihxvxVxSXFWxMd7s9ZPv', 'caller': {'type': 'direct'}, 'input': {}, 'name': 'get_products', 'type': 'tool_use', 'toolset_name': None}]
Tool Calls:
  get_products (toolu_01CdihxvxVxSXFWxMd7s9ZPv)
 Call ID: toolu_01CdihxvxVxSXFWxMd7s9ZPv
  Args:


In [31]:
response.tool_calls

[{'name': 'get_products',
  'args': {},
  'id': 'toolu_01CdihxvxVxSXFWxMd7s9ZPv',
  'type': 'tool_call'}]

In [ ]:
# Ejemplo que no debería llamr tool

messages = [
    SystemMessage(content=system_prompt),
    HumanMessage(content="Hola, ¿cómo estás?"),
]

reponse = llm_tools.invoke(messages)
reponse.pretty_print()


================================== Ai Message ==================================

¡Hola! Estoy muy bien, gracias por preguntar. 😊

Soy tu asistente de ventas de la tienda de ropa, y estoy aquí para ayudarte. Puedo:

1. **Mostrar los productos disponibles** - Te puedo mostrar la lista de prendas que tenemos en la tienda
2. **Informarte sobre el clima** - Puedo decirte cómo está el clima en cualquier ciudad para que elijas la ropa más adecuada

¿Hay algo en lo que pueda ayudarte hoy? ¿Buscas alguna prenda en particular o quieres saber el clima de alguna ciudad?


In [68]:
# Example with tool call and args
# Here the agent infer Bogotá is the capital of Colombia -> here is the power of the LLM to infer and use tools

messages = [
    SystemMessage(content=system_prompt),
    HumanMessage(content="¿cuál es el clima en la capital de Colombia?"),
]

reponse = llm_tools.invoke(messages)
reponse.pretty_print()


================================== Ai Message ==================================

[{'id': 'toolu_01U5Jv8obzBuQT68tbHQw4Hf', 'caller': {'type': 'direct'}, 'input': {'city': 'Bogotá'}, 'name': 'get_weather', 'type': 'tool_use', 'toolset_name': None}]
Tool Calls:
  get_weather (toolu_01U5Jv8obzBuQT68tbHQw4Hf)
 Call ID: toolu_01U5Jv8obzBuQT68tbHQw4Hf
  Args:
    city: Bogotá


In [69]:
# El LLM ya pidió usar get_weather.

reponse.tool_calls[0]

{'name': 'get_weather',
 'args': {'city': 'Bogotá'},
 'id': 'toolu_01U5Jv8obzBuQT68tbHQw4Hf',
 'type': 'tool_call'}

Paper de ReAct: https://arxiv.org/pdf/2210.03629

In [ ]:
# Simular el ciclo ReAct: LLM -> tools -> LLM
# LLM solicita (tool_use) -> Python ejecuta -> Python devuelve (tool_result) -> LLM responde.

from langchain.messages import ToolMessage

messages = [
    SystemMessage(content=system_prompt),
    HumanMessage(content="¿cuál es el clima en la capital de Colombia?"),
]

reponse = llm_tools.invoke(messages)
reponse.pretty_print()


================================== Ai Message ==================================

[{'id': 'toolu_01YRK2evJQ3XsKcmSC1GeHPm', 'caller': {'type': 'direct'}, 'input': {'city': 'Bogotá'}, 'name': 'get_weather', 'type': 'tool_use', 'toolset_name': None}]
Tool Calls:
  get_weather (toolu_01YRK2evJQ3XsKcmSC1GeHPm)
 Call ID: toolu_01YRK2evJQ3XsKcmSC1GeHPm
  Args:
    city: Bogotá


In [ ]:
# 1. Guardar la solicitud del LLM.
messages.append(reponse)

for message in messages:
    message.pretty_print()

================================ System Message ================================


Eres un asistente de ventas de una tienda de ropa. Tu tarea es ayudar a los clientes a encontrar los productos que buscan y proporcionar información sobre el clima en diferentes ciudades.

Tus tools son:
- get_products: Devuelve una lista de productos disponibles en la tienda.
- get_weather: Obtiene el clima actual de una ciudad.

================================ Human Message =================================

¿cuál es el clima en la capital de Colombia?
================================== Ai Message ==================================

[{'id': 'toolu_01YRK2evJQ3XsKcmSC1GeHPm', 'caller': {'type': 'direct'}, 'input': {'city': 'Bogotá'}, 'name': 'get_weather', 'type': 'tool_use', 'toolset_name': None}]
Tool Calls:
  get_weather (toolu_01YRK2evJQ3XsKcmSC1GeHPm)
 Call ID: toolu_01YRK2evJQ3XsKcmSC1GeHPm
  Args:
    city: Bogotá


In [74]:
# 2. Ejecutar la tool.
tool_call = reponse.tool_calls[0]
result = get_weather.invoke(tool_call["args"])

In [75]:
# 3. Guardar el resultado.
messages.append(ToolMessage(content=result, tool_call_id=tool_call["id"]))


for message in messages:
    message.pretty_print()

================================ System Message ================================


Eres un asistente de ventas de una tienda de ropa. Tu tarea es ayudar a los clientes a encontrar los productos que buscan y proporcionar información sobre el clima en diferentes ciudades.

Tus tools son:
- get_products: Devuelve una lista de productos disponibles en la tienda.
- get_weather: Obtiene el clima actual de una ciudad.

================================ Human Message =================================

¿cuál es el clima en la capital de Colombia?
================================== Ai Message ==================================

[{'id': 'toolu_01YRK2evJQ3XsKcmSC1GeHPm', 'caller': {'type': 'direct'}, 'input': {'city': 'Bogotá'}, 'name': 'get_weather', 'type': 'tool_use', 'toolset_name': None}]
Tool Calls:
  get_weather (toolu_01YRK2evJQ3XsKcmSC1GeHPm)
 Call ID: toolu_01YRK2evJQ3XsKcmSC1GeHPm
  Args:
    city: Bogotá
================================= Tool Message =================================

E

In [76]:
# 4. Obtener la respuesta del LLM.
reponse = llm_tools.invoke(messages)
reponse.pretty_print()

================================== Ai Message ==================================

El clima en **Bogotá**, la capital de Colombia, es:

- **Temperatura**: 15.8 °C
- **Viento**: 12.9 km/h

Es un clima fresco y con viento moderado. ¿Te gustaría que te ayude a encontrar ropa adecuada para este clima?


In [77]:
messages.append(reponse)

for message in messages:
    message.pretty_print()

================================ System Message ================================


Eres un asistente de ventas de una tienda de ropa. Tu tarea es ayudar a los clientes a encontrar los productos que buscan y proporcionar información sobre el clima en diferentes ciudades.

Tus tools son:
- get_products: Devuelve una lista de productos disponibles en la tienda.
- get_weather: Obtiene el clima actual de una ciudad.

================================ Human Message =================================

¿cuál es el clima en la capital de Colombia?
================================== Ai Message ==================================

[{'id': 'toolu_01YRK2evJQ3XsKcmSC1GeHPm', 'caller': {'type': 'direct'}, 'input': {'city': 'Bogotá'}, 'name': 'get_weather', 'type': 'tool_use', 'toolset_name': None}]
Tool Calls:
  get_weather (toolu_01YRK2evJQ3XsKcmSC1GeHPm)
 Call ID: toolu_01YRK2evJQ3XsKcmSC1GeHPm
  Args:
    city: Bogotá
================================= Tool Message =================================

E